In [1]:
# Check GPU

import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [2]:
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install pandas
!pip install numpy

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

# Load datasets
fake_df = pd.read_csv('/content/drive/MyDrive/FakeNewsDetector/data/Fake.csv')
true_df = pd.read_csv('/content/drive/MyDrive/FakeNewsDetector/data/True.csv')

# Add labels
fake_df['label'] = 0
true_df['label'] = 1

# Combine datasets
df = pd.concat([fake_df, true_df], axis=0)

# Shuffle dataset
df = df.sample(frac=1).reset_index(drop=True)

# Show dataset info
print(df.head())

print("\nDataset Shape:")
print(df.shape)

                                               title  \
0  MICHELLE OBAMA Slams America At Iranian Party:...   
1  U.S., Iran discuss fulfilling nuclear deal ple...   
2  British lawmakers debate banning Trump after M...   
3      Delta to cancel about 800 flights due to Irma   
4   Jimmy Fallon’s Trump Claims VICTORY As ‘The W...   

                                                text       subject  \
0  So you have to listen to this little speech by...     left-news   
1  UNITED NATIONS (Reuters) - U.S. Secretary of S...  politicsNews   
2  LONDON (Reuters) - British lawmakers on Monday...  politicsNews   
3  (Reuters) - Delta Air Lines Inc said it would ...     worldnews   
4  It can be said that Donald Trump was less than...          News   

                  date  label  
0          Apr 9, 2016      0  
1      April 19, 2016       1  
2    January 18, 2016       1  
3  September 11, 2017       1  
4     February 4, 2016      0  

Dataset Shape:
(44898, 5)


In [24]:
from sklearn.model_selection import train_test_split
df['content'] = df['title'] + " " + df['text']
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 35918
Testing samples: 8980


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(35918, 111925)


In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [27]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9870824053452116


In [47]:
news = ["Suraj would become the prime minister of India in 2029."]

news_vector = vectorizer.transform(news)

prediction = model.predict(news_vector)

print(prediction)

if prediction[0] == 0:
    print("FAKE NEWS")
else:
    print("REAL NEWS")

[1]
REAL NEWS


In [48]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))

print("\n")

print(classification_report(y_test, y_pred))

[[4622   62]
 [  54 4242]]


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4684
           1       0.99      0.99      0.99      4296

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [49]:
import pickle

# Save model
pickle.dump(model, open('fake_news_model.pkl', 'wb'))

# Save vectorizer
pickle.dump(vectorizer, open('tfidf_vectorizer.pkl', 'wb'))

print("Model Saved Successfully")

Model Saved Successfully


In [50]:
# Load saved files

loaded_model = pickle.load(open('fake_news_model.pkl', 'rb'))
loaded_vectorizer = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))

# Test custom news

news = ["India launches new renewable energy initiative"]

news_vector = loaded_vectorizer.transform(news)

prediction = loaded_model.predict(news_vector)

if prediction[0] == 0:
    print("FAKE NEWS")
else:
    print("REAL NEWS")

REAL NEWS


In [51]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [52]:
from sklearn.model_selection import train_test_split

# Use combined content
df['content'] = df['title'] + " " + df['text']

X = df['content'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [53]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [54]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256
)

In [55]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [56]:
train_dataset = NewsDataset(train_encodings, y_train)
test_dataset = NewsDataset(test_encodings, y_test)

In [57]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [58]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [59]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss
10,0.682752
20,0.570350
30,0.438758
40,0.368285
50,0.202027
60,0.083847
70,0.033509
80,0.010933
90,0.004017
100,0.002332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4490, training_loss=0.013675330907444367, metrics={'train_runtime': 2621.8648, 'train_samples_per_second': 13.699, 'train_steps_per_second': 1.713, 'total_flos': 4725211443210240.0, 'train_loss': 0.013675330907444367, 'epoch': 1.0})

In [72]:
import torch

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model.to(device)

news = ["Modi becomes the Prime Minister of India in 2029"]

inputs = tokenizer(
    news,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

# Move inputs to device
inputs = {key: value.to(device) for key, value in inputs.items()}

# Prediction
outputs = model(**inputs)

prediction = torch.argmax(outputs.logits).item()

if prediction == 0:
    print("FAKE NEWS")
else:
    print("REAL NEWS")

FAKE NEWS


In [73]:
model.save_pretrained("bert_fake_news_model")
tokenizer.save_pretrained("bert_fake_news_model")

print("BERT Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT Model Saved Successfully


In [74]:
from transformers import BertTokenizer, BertForSequenceClassification

loaded_tokenizer = BertTokenizer.from_pretrained("bert_fake_news_model")

loaded_model = BertForSequenceClassification.from_pretrained(
    "bert_fake_news_model"
)

news = ["Scientists discover aliens inside volcano"]

inputs = loaded_tokenizer(
    news,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

outputs = loaded_model(**inputs)

prediction = torch.argmax(outputs.logits).item()

if prediction == 0:
    print("FAKE NEWS")
else:
    print("REAL NEWS")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FAKE NEWS


In [75]:
!zip -r bert_fake_news_model.zip bert_fake_news_model

  adding: bert_fake_news_model/ (stored 0%)
  adding: bert_fake_news_model/tokenizer.json (deflated 71%)
  adding: bert_fake_news_model/model.safetensors (deflated 7%)
  adding: bert_fake_news_model/config.json (deflated 53%)
  adding: bert_fake_news_model/tokenizer_config.json (deflated 42%)
